## 1) Setup

In [31]:

# If needed, install once:
# %pip install --upgrade sentence-transformers numpy pandas tqdm
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb

### PreDev Setup

In [32]:
from importlib import reload  # Reload modules during development
import os  # OS utilities
import requests  # HTTP requests
import numpy as np  # Numerical operations
import faiss  # Vector similarity search

import database  # Local database module
from database import AmberChromaAPI  # Amber-Chroma interface

from pypdf import PdfReader  # PDF reading
from sentence_transformers import SentenceTransformer  # Text embeddings

reload(database)  # Refresh module changes


<module 'database' from '/home/bsauce11/RAG_Prototype/Code_Saucedo/My_PreDev/database.py'>

In [33]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

### Document Splitting

In [34]:
# # Initialize text splitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,  # Maximum size of each chunk
#     chunk_overlap=50,  # Overlap between chunks to maintain context
#     length_function=len,
#     separators=[" "]  # Hierarchy of separators
# )
# chunks=text_splitter.split_documents(documents)
#
# print(f"Created {len(chunks)} chunks from {len(documents)} documents")
# print(f"\nChunk example:")
# print(f"Content: {chunks[0].page_content[:150]}...")
# print(f"Metadata: {chunks[0].metadata}")

In [35]:
# chunks

### Embedding Models

In [36]:
### Huggingface model

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [37]:
# vector=embeddings.embed_query(sample_text)
# vector

### Amber ChromaDB

In [38]:
# Chroma DB instance
API_CHROMA_DB = AmberChromaAPI(db_path="/opt/chromadb/data/prompt_db")
# Embedding model
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
# PDF file path
PDF_ADDRESS = "Amber25.pdf"
# Local Ollama server URL
OLLAMA_URL = "http://127.0.0.1:11434"


Using local ChromaDB path: /opt/chromadb/data/prompt_db


In [39]:
threshold_ChromaDB = 0.35 # Similarity threshold for Mails
threshold_PDF = 0.45  # Similarity threshold for PDF results

### Chunking

In [40]:
# # Retrieve relevant chunks from hybrid retriever
# chunks = retrieve_with_pdf(
#     question,
#     k_chroma=50,     # Number of ChromaDB results
#     k_pdf=5,         # Number of PDF results
#     threshold=threshold_ChromaDB    # Similarity threshold
# )

In [41]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# If you already have this client, reuse it:
client = chromadb.PersistentClient(path="/opt/chromadb/data/prompt_db")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    client=client,
    collection_name="rag_collection",
    embedding_function=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [42]:
# ## Create a Chromdb vector store
# persist_directory="/opt/chromadb/data/prompt_db"
#
# ## Initialize Chromadb with HuggingFace embeddings
# vectorstore=Chroma.from_documents(
#     documents=chunks,
#     embedding=HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"),
#     persist_directory=persist_directory,
#     collection_name="rag_collection"
#
# )
#
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Vector store name: {vectorstore._collection.name}")

Vector store created with 0 vectors
Vector store name: rag_collection


### Test Similarity Search

In [43]:
query="What is Amber?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[]

In [44]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is Amber?

Top 0 similar chunks:


### Advanced Similarity Search With Scores

In [45]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [46]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

ChatOllama(model='llama3.1:8b', temperature=0.0)

In [47]:
# from langchain_community.llms import Ollama
#
# #OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
# #OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
#
# llm = Ollama(model="llama3.1:8b")

In [48]:
# Use the LLM to generate prompt
test_response=llm.invoke("What is Amber?")
test_response

AIMessage(content="Amber is a fascinating substance with a rich history. Here's what it is:\n\n**Definition:** Amber is a fossilized tree resin that has been hardened over time, often containing ancient plant and animal remains.\n\n**Formation:** Amber forms when trees produce resin as a defense mechanism against injury or infection. This sticky, viscous liquid flows out of the tree and hardens in the presence of oxygen, forming a solid, transparent to yellowish-brown substance.\n\n**Composition:** Amber is primarily composed of organic compounds, such as terpenes, waxes, and other hydrocarbons. It can also contain small amounts of minerals like silica or calcium carbonate.\n\n**Properties:** Amber has several unique properties:\n\n1. **Fragrance**: Amber often retains the scent of the original resin, which can be pleasant and aromatic.\n2. **Transparency**: Amber is typically transparent to translucent, allowing light to pass through.\n3. **Hardness**: Amber is relatively hard and dur

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [49]:
# ## Convert vector store to retriever
# retriever=vectorstore.as_retriever(
#      search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
#  )
# retriever

In [50]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [51]:
SYSTEM_PROMPT = """
You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer.
Do not include citations, references, or metadata.
"""

In [52]:
"""
You are a concise, technical assistant for Amber molecular simulation users. "
    Answer the user's question using ONLY the provided context.
    If the answer cannot be determined from the context, say you do not know.
    Cite sources using [Title#chunkN] notation.
    Do not speculate or introduce external knowledge.
"""

'\nYou are a concise, technical assistant for Amber molecular simulation users. "\n    Answer the user\'s question using ONLY the provided context.\n    If the answer cannot be determined from the context, say you do not know.\n    Cite sources using [Title#chunkN] notation.\n    Do not speculate or introduce external knowledge.\n'

In [53]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer.
Do not include citations, references, or metadata.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If the Context contains relevant information, use it to answer as completely as possible.\n5) Do NOT mention an Persona, Identity, or Role in your answer.\n6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.\n7) Do NOT include citation marker

In [54]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x146cfed9c0b0>, search_kwargs={})

In [55]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [56]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x146cfed9c0b0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If t

In [57]:
response=rag_chain_lcel.invoke("What is Amber")
response

'The AMBER (Assisted Model Building with Energy Refinement) molecular dynamics suite is a software package used for simulating the behavior of molecules in various environments. It was originally developed by Peter Kollman and his group at the University of California, San Francisco.\n\nTechnical Explanation:\nAMBER uses a combination of classical mechanics and quantum mechanics to simulate the interactions between atoms and molecules. The software includes tools for building molecular models, calculating energies, and performing simulations using various algorithms such as molecular dynamics (MD) and Monte Carlo (MC). AMBER also includes a parameter set that is widely used in biomolecular simulations.\n\nPractical Guidance:\nTo use AMBER, one typically starts by preparing the input files, including the topology file (.prmtop) and the coordinate file (.crd or .pdb), which describe the molecular structure. The user then runs the simulation using the pmemd or sander executable, specifyin

In [75]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Amber")


In [59]:
#retriever.get_relevant_documents("What is Deep Learning")
retDocs = retriever.invoke("What is Deep Learning")
retDocs

[]

In [60]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [61]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What can I use Amber for?")

Testing LCEL Chain:
Question: What can I use Amber for?
--------------------------------------------------
Answer: Molecular dynamics simulations, free energy calculations, molecular mechanics, normal mode analysis, and structure prediction.

You can use Amber to study protein-ligand interactions, protein folding, and protein-protein interactions. It is also used for simulating the behavior of small molecules, nucleic acids, and lipids.

Source Documents:


In [62]:
query_rag_lcel("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
--------------------------------------------------
Answer: During minimization in AMBER, SHAKE should be disabled because it can lead to an artificial stabilization of the system. SHAKE is a harmonic constraint that fixes all bonds involving hydrogen atoms, which can prevent the system from relaxing and reaching its true minimum energy state.

Technical explanation:
The SHAKE algorithm is used to constrain bond lengths involving hydrogen atoms, which are typically much lighter than other atoms in the molecule. This can lead to an artificial stabilization of the system, as the hydrogen atoms are not allowed to move freely. During minimization, it's essential to allow the system to relax and reach its true minimum energy state, rather than being artificially stabilized by SHAKE.

Practical guidance:
To disable SHAKE during minimization in AMBER, use the following command: `sander -p input.parm -o output.trr -c input.crd

In [63]:
query_rag_lcel("How does AMBER handle time-series anomaly detection?")

Question: How does AMBER handle time-series anomaly detection?
--------------------------------------------------
Answer: AMBER uses the "ptraj" module to analyze trajectory files and detect anomalies in molecular dynamics simulations. The ptraj tool can be used with various analysis tools such as RMSD, RMSF, and others to identify unusual behavior or patterns in the simulation.

Technical Explanation:
The ptraj module reads in a trajectory file generated by AMBER's pmemd or sander engines and allows users to perform various analyses on the molecular dynamics data. Users can specify specific atoms or residues of interest and apply filters to focus on particular regions of the simulation. The ptraj tool then outputs results that can be used for further analysis, including time-series anomaly detection.

Practical Guidance:
To use ptraj for time-series anomaly detection, users should first generate a trajectory file using pmemd or sander. Then, they can run ptraj with specific flags to a

In [76]:
query_rag_lcel("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
--------------------------------------------------
Answer: To get SHAKE to consider two different residue names as water, you need to assign them a common atom type that is associated with water in the AMBER force field. 

In the AMBER parameter file (par_all36_prot.ff or par_all36mio_prot.ff), the atom types for water are defined under the "water" section. You can assign the two residue names to have the same atom type as water by modifying their atom types in the topology file (.top) using the `ATOM` keyword.

For example, if you want to consider residues 'RES1' and 'RES2' as water, you would add the following lines to your topology file:

```
ATOM  RES1  OW   0.00000  0.00000  0.00000
ATOM  RES1  HW1  0.00000  0.00000  0.00000
ATOM  RES2  OW   0.00000  0.00000  0.00000
ATOM  RES2  HW1  0.00000  0.00000  0.00000
```

Then, in your parameter file, you would assign the atom types for these residues to be

In [77]:
query_rag_lcel("How can I add the CG protein into a CG bilayer?")

Question: How can I add the CG protein into a CG bilayer?
--------------------------------------------------
Answer: To add the CG protein into a CG bilayer, you can use the `edit` command in AmberTools. Specifically, you would use the `edit` command to insert the protein molecule into the existing bilayer system.

Technical explanation:
The `edit` command is used to modify or manipulate molecules within an Amber simulation system. In this case, it allows you to add a new molecule (the CG protein) to an existing system (the CG bilayer). This can be done by specifying the coordinates of the protein and the bilayer, and then using the `edit` command to merge them into a single system.

Practical guidance:
To use the `edit` command, you would need to have the coordinates of both the protein and the bilayer in separate files. You would then specify these files as input to the `edit` command, along with any necessary flags or options to control the merging process. For example:

```
edit -i

In [78]:
query_rag_lcel("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
--------------------------------------------------
Answer: To obtain a Z-DNA structure from NAB, you can use the following steps:

1. Run `sander` with the `-ZDNA` flag to generate a Z-DNA structure.
2. Use the `zdnadist` command to calculate the Z-DNA distortion energy.

Technical explanation: The `-ZDNA` flag in `sander` allows for the generation of a Z-DNA structure from an input NAB file. This flag is used in conjunction with other flags, such as `-p` and `-i`, to specify the parameters and input files required for the calculation. The `zdnadist` command can then be used to calculate the distortion energy associated with the generated Z-DNA structure.

Practical guidance: Make sure to include the `-ZDNA` flag in your `sander` command, and specify the correct input file and parameters as needed. Additionally, ensure that you have the necessary files and directories set up for the calculation.

Source Documents:


In [79]:
query_rag_lcel("If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?")

Question: If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?
--------------------------------------------------
Answer: Yes, if the system is not properly neutralized or ions are not added appropriately, it can affect the properties of the system, including planarity.

The planarity of a system in AMBER simulations depends on the accurate representation of electrostatic interactions. Neutralization and ion addition help to maintain the correct electrostatic balance, which is crucial for maintaining the planar geometry of molecules like aromatic rings or peptide planes. If this balance is disrupted, it can lead to deviations from the expected planarity.

To ensure proper neutralization and ion addition, it's essential to follow best practices in AMBER simulations, such as using the `ionize` command to add ions and adjusting the system's charge accordingly. Additio

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [66]:
# from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
# from langchain_core.prompts import MessagesPlaceholder
# from langchain_core.messages import HumanMessage, AIMessage

In [67]:
# ## create a prompt that includes the chat history
# contextualize_q_system_prompt = """Given a chat history and the latest user question
# which might reference context in the chat history, formulate a standalone question
# which can be understood without the chat history. Do NOT answer the question,
# just reformulate it if needed and otherwise return it as is."""
#
# contextualize_q_prompt = ChatPromptTemplate.from_messages([
#     ("system", contextualize_q_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])

In [68]:
# ## create history aware retriever
# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
# history_aware_retriever

In [69]:
# from langchain_classic.chains.retrieval import create_retrieval_chain
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
#
# # Create a new document chain with history
# qa_system_prompt = """You are an assistant for question-answering tasks.
# Use the following pieces of retrieved context to answer the question.
# If you don't know the answer, just say that you don't know.
# Use three sentences maximum and keep the answer concise.
#
# Context: {context}"""
#
# qa_prompt = ChatPromptTemplate.from_messages([
#     ("system", qa_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])
#
# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
#
# # Create conversational RAG chain
# conversational_rag_chain = create_retrieval_chain(
#     history_aware_retriever,
#     question_answer_chain
# )
# print("Conversational RAG chain created!")

In [70]:
# chat_history=[]
# # First question
# result1 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What is machine learning?"
# })
# print(f"Q: What is machine learning?")
# print(f"A: {result1['answer']}")

In [71]:
# chat_history.extend([
#     HumanMessage(content="What is machine learning"),
#     AIMessage(content=result1['answer'])
# ])

In [72]:
# chat_history

In [73]:
# ## Follow up question
# # Follow-up question
# result2 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What are its main types?"  # Refers to ML from previous question
# })
# result2

In [74]:
# result2['answer']